# Assignment 2: Theses

---

## Task 2) Theses Inspiration

Imagine you'd have to write another thesis, and you just can't find a good topic to work on.
Well, n-grams to the rescue!
Download the `theses.txt` data set from the `Supplemental Materials` in the `Files` section of our Microsoft Teams group.
This dataset consists of approx. 1,000 theses topics chosen by students in the past.

In this assignment, you will be sampling from n-grams to generate new potential thesis topics.
Pay extra attention to preprocessing: How would you handle hyphenated words and acronyms/abbreviations?

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [1]:
# Dependencies
import re
import random
import numpy as np
from typing import List, Dict, Union
from nltk.tokenize import word_tokenize

### Prepare the Data

1.1 Spend some time on pre-processing. How would you handle hyphenated words and abbreviations/acronyms?

In [2]:
def load_theses_titles(filepath: str) -> List[str]:
    """Loads all theses titles and returns them as a list."""
    with open(filepath) as file:
        titles = file.readlines()

    return titles

In [3]:
TokenList = List[str]
DataSet = List[TokenList]


def tokenize(title: str) -> TokenList:
    """Tokenizes the thesis title."""
    title = re.sub(r"(\w)-(\w)", r"\1 \2", title)
    title = title.replace("\n", "")
    title = title.replace("-", "")
    title = title.replace("/", "")
    title = title.replace("\"", "")
    title = title.replace(",", "")
    title = title.replace("?", "")
    return word_tokenize(title, language="german")


def preprocess(data: List[str]) -> DataSet:
    """Preprocesses and tokenizes the given theses titles for further use."""
    tokenized_data: DataSet = list(map(tokenize, data))

    return tokenized_data

In [4]:
theses_data = preprocess(load_theses_titles("data/theses.txt"))
print("Theses titles: {}".format(len(theses_data)))

Theses titles: 964


### Train N-gram Models

2.1 Train n-gram models with n = [1, ..., 5]. What about \<s> and \</s>?

In [5]:
# for convenience, we combine the n-gram models into one
NGramModel = Dict[str, Dict[str, float]]


def update_n_gram_model(model: NGramModel, w: str, previous_words: List[str]):
    """Inserts a new finding into an existing model."""
    key = " ".join(previous_words)

    if key not in model:
        model[key] = {}

    if w in model[key]:
        model[key][w] += 1.0
    else:
        model[key][w] = 1.0



def n_grams_in_title(title_tokens: List[str], n: int) -> NGramModel:
    """
    Extracts all possible n-grams from one thesis title and returns them as a
    small model.
    """
    model = {}

    for i in range(n, len(title_tokens)):
        w = title_tokens[i]  # w_(n-N+1) ... w_n
        previous = title_tokens[i - n: i]  # w_(n-N+1) ... w_(n-1)
        update_n_gram_model(model, w, previous)

    return model


def combine_models(model_a: NGramModel, model_b: NGramModel):
    """Adds all findings from model_b into model_a."""
    for key_b in model_b.keys():
        if key_b in model_a:

            for key_a in model_b[key_b].keys():
                if key_a in model_a[key_b]:
                    model_a[key_b][key_a] += model_b[key_b][key_a]
                else:
                    model_a[key_b][key_a] = model_b[key_b][key_a]
        else:
            model_a[key_b] = model_b[key_b]


def calculate_percentages_in_model(model: NGramModel):
    """
    Computes C(w_(n-N+1) ... w_n) / C(w_(n-N+1) ... w_(n-1)) for each model
    entry
    """
    for key in model.keys():
        total_count = float(sum(model[key].values()))

        for sub_key in model[key].keys():
            model[key][sub_key] /= total_count


def build_n_gram_models(n: int, data: DataSet):
    """This method does calculate all n-grams up to the given n."""
    entire_model = {}

    for thesis_title in data:
        m = n_grams_in_title(title_tokens=thesis_title, n=n)
        combine_models(model_a=entire_model, model_b=m)

    calculate_percentages_in_model(entire_model)

    return entire_model

In [6]:
# just for testing
thesis_model = build_n_gram_models(n=2, data=theses_data)

### Generate the Titles

3.1 Write a generator that provides thesis titles of desired length. Please do not use the available `lm.generate` method but write your own.

3.2 How can you incorporate seed words?

3.3 How do you handle </s> tokens (w.r.t. the desired length?)

3.4 If you didn't just copy what nltk's lm.generate does: compare the outputs.

In [7]:
# Notice: If you fix the seed in numpy.random.choice, you get reproducible results.


def get_start_tokens(model: NGramModel, seed: str) -> Union[List[str], None]:
    """Generates a token from a given seed word."""
    possible_tokens = list(filter(lambda s: seed in s, list(model.keys())))

    if len(possible_tokens) == 0:
        return None
    else:
        return np.random.choice(possible_tokens).split(" ")


def sample_next_token(n: int, prev: TokenList, model: NGramModel) -> Union[str, None]:
    """Samples the next word for the given n_grams."""
    key = " ".join(prev[-n:])

    possible_tokens = model.get(key)
    if possible_tokens is None:
        return None

    random_number = random.random()

    for tkn in possible_tokens:
        random_number -= possible_tokens.get(tkn)
        if random_number < 0:
            return tkn


def generate(n: int, model: NGramModel, seed: str, title_length: int):
    """Generates a thesis title using the n_grams, seed word and title length."""
    sampled_tokens = get_start_tokens(model, seed)

    if sampled_tokens is None:
        print("No viable start token was found in the given model.")
        return ""

    for _ in range(title_length):
        next_token = sample_next_token(n, sampled_tokens, model)

        # in case there is no known follow up token in the model
        if next_token is None:
            break

        sampled_tokens.append(next_token)

    return " ".join(sampled_tokens)

In [8]:
# generate some theses titles with seed words and length
n = 2
thesis_model = build_n_gram_models(n=n, data=theses_data)

title_length = 20
seed_word =  "Entwicklung"
print("### Seed word: `{}`".format(seed_word))
for _ in range(5):
    thesis_title = generate(n=n, model=thesis_model, seed=seed_word, title_length=title_length)
    print(thesis_title)

seed_word =  "Cloud"
print("\n### Seed word: `{}`".format(seed_word))
for _ in range(5):
    thesis_title = generate(n=n, model=thesis_model, seed=seed_word, title_length=title_length)
    print(thesis_title)

### Seed word: `Entwicklung`
Software Entwicklung und prototypische Implementierung für die optimale Bearbeitung von Feldern
Entwicklung eines KI basierten Answer Bots zur Lösung von Continental Engineering Services
Fertigungsindustrie Entwicklung und Implementierung eines mehrbenutzer basierten Zugriffssystems für das Verfahren Virtueller Arbeitsmarkt der Bundesagentur für Arbeit
Technische Entwicklung Vorteile und Risiken gibt es Welche rechtlichen bzw. ethischen Aspekte sollten berücksichtigt werden
die Entwicklung und Analyse von EAST ADL Modellen im Hinblick auf Rechte und Freiheiten der Betroffenen

### Seed word: `Cloud`
Cloud Transformation Analyse zur Auswahl einer Digital Adoption Platform für SAP SuccessFactors in der IT Strategien der bayrischen Hochschulen und der Studierendenanforderungen
Cloudplattform mit maschinellen Lernverfahren anhand einer E Learning im Studium und empirischer Vergleich zu Präsenzunterricht am Beispiel des DATEV Schnittstellensystems Pro
Cloud Tran